## Template TICKER analysis

### 0. Setup and readers

In [1]:
TICKER = "VIRTUAL"
TICK = 0.00001

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Ноутбук лежит глубже BacktestingBasics — добавляем research/ в путь, чтобы найти пакет tools.
repo_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                 if (p / ".git").exists())
research_dir = repo_root / "research"
if str(research_dir) not in sys.path:
    sys.path.insert(0, str(research_dir))

from tools import ROOT
from tools.readers.lighter import LobReader, LobReaderRaw, TradesReader, TickerReader, MarketStatsReader
from tools.readers.binance import BookTickerReader, AggTradesReader, MarkPriceReader, LiquidationsReader

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

In [3]:
import plotly.io as pio

pio.renderers.default = "iframe"  # рендер plotly внутри JupyterLab (иначе вывод пустой)

In [4]:
def style_figure(fig, height):
    """Единое оформление для всех графиков ноутбука."""
    fig.update_layout(template="plotly_white", height=height, showlegend=False,
                      bargap=0.02, margin=dict(l=55, r=20, t=50, b=45))
    return fig


def add_percentile_band(fig, row, x, low, median, high, rgb="76,120,168"):
    """Линия медианы и затенённая полоса [low, high] в подграфике (row, 1)."""
    fig.add_trace(go.Scatter(x=x, y=high, mode="lines", line_width=0, hoverinfo="skip"), row, 1)
    fig.add_trace(go.Scatter(x=x, y=low, mode="lines", line_width=0, fill="tonexty",
                             fillcolor=f"rgba({rgb},0.2)", hoverinfo="skip"), row, 1)
    fig.add_trace(go.Scatter(x=x, y=median, mode="lines",
                             line=dict(color=f"rgb({rgb})", width=1.2)), row, 1)


def percentile_summary(series, decimals):
    """Перцентили + mean/max одной серии как dict — кирпичик строки-артефакта."""
    levels = (1, 5, 25, 50, 75, 90, 95, 99)
    summary = {f"p{level}": np.percentile(series, level) for level in levels}
    summary["mean"] = series.mean()
    summary["max"] = series.max()
    return {name: round(float(value), decimals) for name, value in summary.items()}


def build_taker_orders(trades):
    """Тейкер-филлы (type==trade, без ликвидаций) и их агрегация в тейкер-ордера.
    Общее ядро для D (свипы) и B (adverse selection). Возвращает (taker_fills, orders).
    Несём оба движковых времени: t_match (timestamp, матч) и t_commit (transaction_time)."""
    maker_ask = trades["is_maker_ask"].to_numpy()
    is_trade = trades["type"].eq("trade").to_numpy()
    taker_fills = pd.DataFrame({
        "order_id": np.where(maker_ask, trades["bid_id"], trades["ask_id"]),
        "account": np.where(maker_ask, trades["bid_account_id"], trades["ask_account_id"]),
        "buy": maker_ask,
        "price": trades["price"].to_numpy(),
        "notional": (trades["price"] * trades["size"]).to_numpy(),
        "block": trades["block_height"].to_numpy(),
        "t_match": trades["timestamp"].to_numpy(),
        "t_commit": trades["transaction_time"].to_numpy(),
    })[is_trade]

    orders = taker_fills.groupby("order_id").agg(
        t_match=("t_match", "min"),
        t_commit=("t_commit", "min"),
        notional=("notional", "sum"),
        n_legs=("price", "count"),
        n_levels=("price", "nunique"),
        px_min=("price", "min"),
        px_max=("price", "max"),
        px_mean=("price", "mean"),
        block_span=("block", lambda block: block.max() - block.min()),
        buy=("buy", "first"),
    )
    orders["category"] = np.where(orders["block_span"] > 1, "algo_twap",
                                  np.where(orders["n_levels"] >= 2, "sweep", "single_level"))
    orders["depth_bps"] = (orders["px_max"] - orders["px_min"]) / orders["px_mean"] * 1e4
    return taker_fills, orders


def drift_matrix(anchor_us, side, book_ts, book_mid, book_valid, horizons_s):
    """Подписанный дрифт mid по сетке горизонтов относительно якоря anchor_us (µs).
    m₀ = последний снапшот строго до anchor; m_h = последний с временем ≤ anchor+h.
    Возвращает (drift[events×horizons] bps, reach bool, i0 clipped, valid0)."""
    i0 = np.searchsorted(book_ts, anchor_us, side="left") - 1
    valid0 = (i0 >= 0)
    i0 = np.clip(i0, 0, None)
    valid0 = valid0 & book_valid[i0]
    mid0 = book_mid[i0]

    drift = np.full((len(anchor_us), len(horizons_s)), np.nan)
    reach = np.zeros_like(drift, dtype=bool)
    for j, h in enumerate(horizons_s):
        ih = np.clip(np.searchsorted(book_ts, anchor_us + int(h * 1e6), side="right") - 1, 0, len(book_ts) - 1)
        ok = valid0 & (anchor_us + h * 1e6 <= book_ts[-1]) & book_valid[ih]
        drift[:, j] = np.where(ok, side * (book_mid[ih] - mid0) / mid0 * 1e4, np.nan)
        reach[:, j] = ok
    return drift, reach, i0, valid0


def load_streaming(reader_cls, paths, columns=None):
    """Загрузка jsonl-потока ПО ФАЙЛУ: каждый парсится и (опц.) ужимается отдельно, склеиваются
    тонкие куски. Пик = один файл, а не список-словарей на всю неделю (иначе .load() строит млн dict\'ов)."""
    parts = []
    for path in paths:
        frame = reader_cls([path]).load()
        parts.append(frame[columns] if columns is not None else frame)
    return pd.concat(parts, ignore_index=True) if parts else reader_cls(paths).load()

In [5]:
# ── Покрытие по часам. ПРАВИМ ТОЛЬКО ЗДЕСЬ по мере доезда данных. ──
# Только часы, где пишут ОБЕ биржи (Lighter + Binance).
HOURS = [
    (20260700 + day, hour)
    for day in range(14, 19)
    for hour in (range(24) if day < 18 else range(15))
]


def stream_paths(venue, stream, ext):
    paths = [ROOT / f"data/{venue}/{TICKER}/{stream}/{day}/{hour:02d}.{ext}" for day, hour in HOURS]
    return [p for p in paths if p.exists()]   # отсутствующие часы тихо пропускаем — покрытие у символов разное


lob_paths = stream_paths("lighter", "lob", "parquet")
trades_paths = stream_paths("lighter", "trades", "jsonl.zst")
ticker_paths = stream_paths("lighter", "ticker", "jsonl.zst")
market_stats_paths = stream_paths("lighter", "market_stats", "jsonl.zst")

book_ticker_paths = stream_paths("binance", "book_ticker", "jsonl.zst")
agg_trades_paths = stream_paths("binance", "agg_trades", "jsonl.zst")
mark_price_paths = stream_paths("binance", "mark_price", "jsonl.zst")
liquidations_paths = stream_paths("binance", "liquidations", "jsonl.zst")

print(f"{TICKER}: заявлено {len(HOURS)}h ({HOURS[0]} … {HOURS[-1]}) | на диске: "
      f"lob {len(lob_paths)}, trades {len(trades_paths)}, ticker {len(ticker_paths)}, "
      f"book_ticker {len(book_ticker_paths)}")

VIRTUAL: заявлено 111h ((20260714, 0) … (20260718, 14)) | на диске: lob 111, trades 110, ticker 111, book_ticker 111


In [6]:
# Только top-of-book (6 колонок) вместо всех 122: parquet проецирует на чтении, RAM ×20 меньше.
# Полную 30-уровневую глубину читает лишь потоковый проход в секции плотности (dens-prep).
TOB_COLUMNS = ["sent_ts", "recv_ts", "ask_px_1", "bid_px_1", "ask_sz_1", "bid_sz_1"]
book = LobReader(lob_paths, columns=TOB_COLUMNS).load()

span_min = (book["recv_ts"].iloc[-1] - book["recv_ts"].iloc[0]) / 1e6 / 60
print(f"lob snapshots: {len(book):,} | span {span_min:.1f} min "
      f"| recv_ts monotonic: {book['recv_ts'].is_monotonic_increasing}")
book.head()

lob snapshots: 1,882,508 | span 6660.0 min | recv_ts monotonic: True


,sent_ts,recv_ts,ask_px_1,bid_px_1,ask_sz_1,bid_sz_1
0,1783987200011000,1783987200090027,0.54905,0.54827,14727.4,57.3
1,1783987200066000,1783987200213904,0.54912,0.54827,17748.5,57.3
2,1783987200116000,1783987200265156,0.54912,0.54827,17748.5,57.3
3,1783987200167000,1783987200281873,0.54912,0.54827,17748.5,55.7
4,1783987200218000,1783987200349793,0.54912,0.54827,17748.5,55.7


In [7]:
# Sanity: вбитый вручную TICK совпадает с фактическим шагом ценовой сетки (ловит опечатку).
# Считаем по L1 обеих сторон: в TOB-проекции нет L2..30, но за неделю L1 покрывает достаточно цен.
grid_prices = np.concatenate([book["ask_px_1"].to_numpy(), book["bid_px_1"].to_numpy()])
grid_prices = grid_prices[grid_prices > 0]
price_grid = int(np.gcd.reduce(np.diff(np.unique(np.round(grid_prices * 1e8).astype(np.int64))))) / 1e8
assert abs(price_grid - TICK) < 1e-12, f"{TICKER}: TICK={TICK} != price grid {price_grid} — поправь TICK"
print(f"TICK {TICK} confirmed vs price grid (L1)")

TICK 1e-05 confirmed vs price grid (L1)


In [8]:
trades = TradesReader(trades_paths).load()

# Ридер сохраняет порядок поступления; внутри батча сделки не отсортированы по времени.
print(f"trades: {len(trades):,} fills | timestamp monotonic: {trades['timestamp'].is_monotonic_increasing}")
trades.head()

trades: 3,267 fills | timestamp monotonic: False


,receive_timestamp,timestamp,transaction_time,trade_id,type,price,size,is_maker_ask,ask_account_id,bid_account_id,ask_id,bid_id,taker_fee,maker_fee,taker_position_size_before,maker_position_size_before,block_height
0,1783987825822193,1783987825765000,1783987825817508,25267331706,trade,0.54703,244.9,True,151511,281474976710654,11821949241120451,12103423791780359,196,0,107949.9,244.9,290650873
1,1783988139357202,1783988139348000,1783988139349986,25267907128,trade,0.54891,15.2,True,281474976710654,721165,11821949241006917,12103423791775663,0,28,-15.2,108194.8,290659193
2,1783989086863052,1783988139348000,1783988139349986,25267907128,trade,0.54891,15.2,True,281474976710654,721165,11821949241006917,12103423791775663,0,28,-15.2,108194.8,290659193
3,1783989086863052,1783987825765000,1783987825817508,25267331706,trade,0.54703,244.9,True,151511,281474976710654,11821949241120451,12103423791780359,196,0,107949.9,244.9,290650873
4,1783989086863052,1783984226250000,1783984226279815,25263895034,trade,0.54351,15.2,False,721165,281474976710654,11821949241099590,12103423791793911,0,28,0.0,107934.7,290614754


In [9]:
BBO_COLUMNS = ["timestamp", "receive_timestamp", "ask_price", "bid_price"]
tickers = load_streaming(TickerReader, ticker_paths, BBO_COLUMNS)   # по файлу + только BBO

# Событийный поток BBO: last_updated_at — движковые µs (в отличие от ms-эмиссии lob.sent_ts).
print(f"ticker: {len(tickers):,} msgs | vs lob: {len(book):,} снапшотов "
      f"| recv monotonic: {tickers['receive_timestamp'].is_monotonic_increasing}")
tickers.head()

ticker: 464,368 msgs | vs lob: 1,882,508 снапшотов | recv monotonic: True


,timestamp,receive_timestamp,ask_price,bid_price
0,1783987200038000,1783987200151620,0.54912,0.54827
1,1783987200158000,1783987200275999,0.54912,0.54827
2,1783987200340000,1783987200389496,0.54912,0.54828
3,1783987200369000,1783987200393479,0.54912,0.54825
4,1783987200411000,1783987200414445,0.54930,0.54825


In [10]:
book_ticker = load_streaming(BookTickerReader, book_ticker_paths, BBO_COLUMNS)   # по файлу + только BBO

# BBO лидера. timestamp = event time биржи (E), last_updated_at = transaction time (T).
print(f"binance book_ticker: {len(book_ticker):,} msgs")
book_ticker.head()

binance book_ticker: 4,671,399 msgs


,timestamp,receive_timestamp,ask_price,bid_price
0,1783987200008000,1783987200043143,0.5493,0.5492
1,1783987200010000,1783987200043563,0.5493,0.5492
2,1783987200010000,1783987200043706,0.5493,0.5492
3,1783987200125000,1783987200295701,0.5493,0.5492
4,1783987200126000,1783987200299682,0.5493,0.5492


In [11]:
market_stats = load_streaming(MarketStatsReader, market_stats_paths)
agg_trades = load_streaming(AggTradesReader, agg_trades_paths)
mark_price = load_streaming(MarkPriceReader, mark_price_paths)

print(f"lighter market_stats: {len(market_stats):,} | "
      f"binance agg_trades: {len(agg_trades):,} | binance mark_price: {len(mark_price):,}")

lighter market_stats: 232,695 | binance agg_trades: 336,921 | binance mark_price: 399,597


### 1. Flows analysis

In [12]:
# Окно: сдвиг от начала записи + длительность. Двигаем этими двумя числами.
# ВАЖНО: все ряды строим по времени ОТПРАВКИ биржей (lob.sent_ts, ticker.timestamp,
# binance.timestamp) — не по времени получения. Иначе сдвиг = наша сеть, а не рынок.
WINDOW_START_S = 5
WINDOW_SECONDS = 30

t_start = int(book["sent_ts"].iloc[0]) + WINDOW_START_S * 1_000_000
t_end = t_start + WINDOW_SECONDS * 1_000_000

lob_win = book[(book["sent_ts"] >= t_start) & (book["sent_ts"] < t_end)]
ticker_win = tickers[(tickers["timestamp"] >= t_start) & (tickers["timestamp"] < t_end)]
binance_win = book_ticker[(book_ticker["timestamp"] >= t_start) & (book_ticker["timestamp"] < t_end)]

ask, bid = lob_win["ask_px_1"], lob_win["bid_px_1"]
spread_ticks = (ask - bid) / TICK

# Ось Y — в тиках Lighter относительно якоря: зазор bid↔ask читается как спред в тиках.
# Binance кладём в ту же шкалу (его тик крупнее, поэтому его котировки «ступают» реже).
anchor = round(((ask.iloc[0] + bid.iloc[0]) / 2) / TICK) * TICK
def in_ticks(price):
    return (price - anchor) / TICK

print(f"в окне: lob {len(lob_win):,} | ticker {len(ticker_win):,} | binance {len(binance_win):,} сообщений")
print(f"anchor {anchor:.6g} | spread (lighter), ticks: p50={spread_ticks.median():.0f} "
      f"min={spread_ticks.min():.0f} max={spread_ticks.max():.0f}")

book_fig = go.Figure()
book_fig.add_trace(go.Scatter(x=pd.to_datetime(lob_win["sent_ts"], unit="us"), y=in_ticks(ask),
                              name="lighter lob ask", mode="lines",
                              line=dict(color="#E45756", width=1.2, shape="hv")))
book_fig.add_trace(go.Scatter(x=pd.to_datetime(lob_win["sent_ts"], unit="us"), y=in_ticks(bid),
                              name="lighter lob bid", mode="lines", fill="tonexty",
                              fillcolor="rgba(76,120,168,0.12)",
                              line=dict(color="#4C78A8", width=1.2, shape="hv")))
book_fig.add_trace(go.Scatter(x=pd.to_datetime(ticker_win["timestamp"], unit="us"),
                              y=in_ticks(ticker_win["ask_price"]), name="lighter ticker ask",
                              mode="markers", marker=dict(color="#E45756", size=5, symbol="circle-open")))
book_fig.add_trace(go.Scatter(x=pd.to_datetime(ticker_win["timestamp"], unit="us"),
                              y=in_ticks(ticker_win["bid_price"]), name="lighter ticker bid",
                              mode="markers", marker=dict(color="#4C78A8", size=5, symbol="circle-open")))
book_fig.add_trace(go.Scatter(x=pd.to_datetime(binance_win["timestamp"], unit="us"),
                              y=in_ticks(binance_win["ask_price"]), name="binance ask", mode="lines",
                              line=dict(color="#F58518", width=1, shape="hv")))
book_fig.add_trace(go.Scatter(x=pd.to_datetime(binance_win["timestamp"], unit="us"),
                              y=in_ticks(binance_win["bid_price"]), name="binance bid", mode="lines",
                              line=dict(color="#72B7B2", width=1, shape="hv")))
style_figure(book_fig, 460).update_layout(
    showlegend=True,
    title_text=f"{TICKER} · L1 по биржевым меткам отправки: lighter lob/ticker vs binance, окно {WINDOW_SECONDS}s")
book_fig.update_yaxes(title_text=f"ticks (lighter) from {anchor:.6g}")
book_fig.show()

в окне: lob 280 | ticker 113 | binance 846 сообщений
anchor 0.54895 | spread (lighter), ticks: p50=78 min=59 max=122


#### Как часто биржа публикует апдейты в каждом потоке?

Меряем интервалы между **моментами публикации биржей** (только биржевые часы, не приёмные —
нас интересует темп самой биржи, а не задержка сети).

In [13]:
def publish_intervals_ms(frame, time_column):
    """Интервалы (мс) между моментами, когда биржа публикует апдейт в потоке.

    Одна публикация = одна уникальная биржевая метка времени. Записи с одинаковой
    меткой схлопываются (ms-квантование; батч сделок в одном сообщении) — иначе
    получили бы фиктивные нулевые интервалы.
    """
    stamps = np.unique(frame[time_column].to_numpy())   # np.unique = сортировка + дедуп
    return np.diff(stamps) / 1000.0                     # µs -> ms

In [14]:
# Реестр потоков: у каждого — метка ПУБЛИКАЦИИ биржей и метка НАШЕГО ПРИЁМА.
#   lighter lob          — sent_ts: единственная биржевая метка книги (эмиссия, ms)
#   lighter trades       — transaction_time: у сообщения трейдов нет метки эмиссии,
#                          ближайшее биржевое время публикации — коммит транзакции (µs)
#   lighter ticker/stats — timestamp: время эмиссии сообщения биржей (ms)
#   binance *            — timestamp: event time биржи (поле E, ms)
STREAMS = [
    # (имя,                 датафрейм,    публикация биржей,  наш приём)
    ("lighter lob",          book,         "sent_ts",          "recv_ts"),
    ("lighter trades",       trades,       "transaction_time", "receive_timestamp"),
    ("lighter ticker",       tickers,      "timestamp",        "receive_timestamp"),
    ("lighter market_stats", market_stats, "timestamp",        "receive_timestamp"),
    ("binance book_ticker",  book_ticker,  "timestamp",        "receive_timestamp"),
    ("binance agg_trades",   agg_trades,   "timestamp",        "receive_timestamp"),
    ("binance mark_price",   mark_price,   "timestamp",        "receive_timestamp"),
]

In [15]:
def publish_stats(frame, time_column):
    """Строка таблицы: сколько апдейтов публикует биржа и с какими интервалами (мс)."""
    gaps = publish_intervals_ms(frame, time_column)
    span_s = gaps.sum() / 1000
    return {
        # "clock": time_column,
        # "n_rows": len(frame),
        # "n_updates": len(gaps) + 1,
        # "updates/s": round((len(gaps) + 1) / span_s, 2) if span_s > 0 else np.nan,
        "p50": round(float(np.percentile(gaps, 50)), 1),
        "p90": round(float(np.percentile(gaps, 90)), 1),
        "p99": round(float(np.percentile(gaps, 99)), 1),
        "mean": round(float(gaps.mean()), 1),
    }


publish_table = pd.DataFrame([publish_stats(frame, publish_col) for _, frame, publish_col, _ in STREAMS],
                             index=[name for name, *_ in STREAMS])
print("Интервалы между публикациями биржи, мс")
publish_table

Интервалы между публикациями биржи, мс


,p50,p90,p99,mean
lighter lob,100.0,501.0,1453.0,212.3
lighter trades,30080.2,359590.6,1541357.5,225513.0
lighter ticker,136.0,1559.0,13772.7,863.4
lighter market_stats,1002.0,3005.0,9561.1,1717.3
binance book_ticker,5.0,368.0,1675.0,129.1
binance agg_trades,327.0,3699.0,11552.0,1303.3
binance mark_price,1000.0,1002.0,1008.0,1000.0


#### Какая средняя задержка получения данных?

Задержка = **наш приём − публикация биржей**, по каждой записи. Показывает, насколько
устаревшей до нас доезжает картинка каждого потока.

In [16]:
def receive_latency_ms(frame, publish_column, receive_column):
    """Задержка получения (мс): наш приём минус момент публикации биржей — по каждой записи.

    Это разность часов ДВУХ машин: в неё входит не только сеть+наш стек, но и рассинхрон
    наших часов с биржевыми (NTP-скью). Отрицательный min — прямой признак скью.
    """
    return (frame[receive_column].to_numpy() - frame[publish_column].to_numpy()) / 1000.0

In [17]:
def latency_stats(frame, publish_column, receive_column):
    """Строка таблицы задержек получения (мс)."""
    lag = receive_latency_ms(frame, publish_column, receive_column)
    return {
        # "publish clock": publish_column,
        # "n": len(lag),
        "min": round(float(lag.min()), 1),
        "p50": round(float(np.percentile(lag, 50)), 1),
        "p90": round(float(np.percentile(lag, 90)), 1),
        "p99": round(float(np.percentile(lag, 99)), 1),
        "mean": round(float(lag.mean()), 1),
        "max": round(float(lag.max()), 1),
    }


latency_table = pd.DataFrame(
    [latency_stats(frame, publish_col, receive_col) for _, frame, publish_col, receive_col in STREAMS],
    index=[name for name, *_ in STREAMS],
)
print("Задержка получения = наш приём − публикация биржей, мс")
latency_table

Задержка получения = наш приём − публикация биржей, мс


,min,p50,p90,p99,mean,max
lighter lob,1.2,2.5,3.7,158.3,8.3,1943.0
lighter trades,2.4,4.5,3992674.1,248185736.7,5890563.2,643205945.0
lighter ticker,1.2,2.6,3.9,156.4,8.8,1688.8
lighter market_stats,1.4,3.9,7.0,163.5,9.8,1476.6
binance book_ticker,0.4,2.0,8.0,133.8,11.6,4087.5
binance agg_trades,0.4,1.5,2.5,39.0,3.3,1713.3
binance mark_price,39.2,69.0,83.5,101.1,69.1,1570.7


#### Насколько цена на Lighter отстаёт от Binance?

Меряем по **нашим часам приёма** (`receive_timestamp` обоих потоков): обе метки ставит
одна машина → между биржами нет NTP-скью, и это ровно та величина, которую переживает
стратегия — «Binance дёрнулся, сколько у меня есть времени до движения Lighter».

Работаем с **изменениями**, не уровнями: базис (перп-премия) между венью на разностях
сокращается. Ряды BBO: Binance `book_ticker`, Lighter `ticker` (событийный — меняется
ровно тогда, когда меняется тач).

In [18]:
# Общий каркас: BBO-ряды обеих бирж на НАШИХ часах приёма + события «Binance сдвинулся».
EVENT_BPS = 1.0        # порог движения Binance (1 тик Binance ≈ 0.48 bps → берём выше шума сетки)
REFRACTORY_MS = 500    # одна волна движения = одно событие, а не десяток


def bbo_series(frame, time_column):
    """(ts, mid, valid) из потока BBO, отсортированные по времени."""
    ts = frame[time_column].to_numpy()
    mid = ((frame["ask_price"] + frame["bid_price"]) / 2).to_numpy()
    valid = (frame["ask_price"].to_numpy() > 0) & (frame["bid_price"].to_numpy() > 0)
    order = np.argsort(ts, kind="stable")
    return ts[order], mid[order], valid[order]


def binance_move_events(ts, mid, threshold_bps, refractory_ms):
    """Индексы апдейтов Binance, где mid сдвинулся на ≥ порога относительно предыдущего."""
    step_bps = np.diff(mid) / mid[:-1] * 1e4
    candidates = np.where(np.abs(step_bps) >= threshold_bps)[0] + 1
    kept, last_ts = [], -np.inf
    for i in candidates:
        if ts[i] - last_ts >= refractory_ms * 1000:
            kept.append(i)
            last_ts = ts[i]
    return np.array(kept, dtype=int)


binance_ts, binance_mid, binance_valid = bbo_series(book_ticker, "receive_timestamp")
lighter_ts, lighter_mid, lighter_valid = bbo_series(tickers, "receive_timestamp")

event_idx = binance_move_events(binance_ts, binance_mid, EVENT_BPS, REFRACTORY_MS)
event_ts = binance_ts[event_idx]
event_step = binance_mid[event_idx] - binance_mid[event_idx - 1]
event_sign = np.sign(event_step)
event_bps = np.abs(event_step) / binance_mid[event_idx - 1] * 1e4

print(f"событий Binance (|Δmid| ≥ {EVENT_BPS} bps, refractory {REFRACTORY_MS} мс): {len(event_idx):,}")
print(f"размер события, bps: p50={np.percentile(event_bps, 50):.2f} "
      f"p90={np.percentile(event_bps, 90):.2f} max={event_bps.max():.2f}")

событий Binance (|Δmid| ≥ 1.0 bps, refractory 500 мс): 46,441
размер события, bps: p50=1.62 p90=1.82 max=47.77


##### A. Кривая отклика: какую долю движения Binance Lighter уже впитал к моменту h

In [19]:
# Отклик обеих бирж на событие. Та же машинерия drift_matrix: m₀ — строго ДО события,
# m_h — последнее известное состояние на момент t+h. Знак — по направлению движения Binance.
HORIZONS_S = [0.0, 0.025, 0.05, 0.1, 0.2, 0.35, 0.5, 1.0, 2.0, 5.0]

lighter_resp, _, _, _ = drift_matrix(event_ts, event_sign, lighter_ts, lighter_mid, lighter_valid, HORIZONS_S)
binance_resp, _, _, _ = drift_matrix(event_ts, event_sign, binance_ts, binance_mid, binance_valid, HORIZONS_S)

horizons_ms = np.array(HORIZONS_S) * 1000
response = pd.DataFrame({
    "binance_bps": np.nanmean(binance_resp, axis=0),
    "lighter_bps": np.nanmean(lighter_resp, axis=0),
}, index=[f"{int(h)}ms" for h in horizons_ms])
response["absorbed"] = response["lighter_bps"] / response["binance_bps"]


def crossing_time_ms(horizons_ms, absorbed, level):
    """Момент (мс), когда доля впитанного впервые достигает level. Линейная интерполяция."""
    above = np.where(absorbed >= level)[0]
    if len(above) == 0:
        return np.nan
    k = int(above[0])
    if k == 0:
        return float(horizons_ms[0])
    x0, x1 = horizons_ms[k - 1], horizons_ms[k]
    y0, y1 = absorbed[k - 1], absorbed[k]
    return float(x0 + (level - y0) * (x1 - x0) / (y1 - y0))


absorbed = response["absorbed"].to_numpy()
half_ms = crossing_time_ms(horizons_ms, absorbed, 0.5)
p90_ms = crossing_time_ms(horizons_ms, absorbed, 0.9)
print(f"ОТВЕТ A: Lighter впитывает 50% движения Binance за {half_ms:.0f} мс, 90% — за {p90_ms:.0f} мс")

resp_fig = go.Figure()
resp_fig.add_trace(go.Scatter(x=horizons_ms, y=response["binance_bps"], name="binance",
                              mode="lines+markers", line=dict(color="#F58518")))
resp_fig.add_trace(go.Scatter(x=horizons_ms, y=response["lighter_bps"], name="lighter",
                              mode="lines+markers", line=dict(color="#4C78A8")))
style_figure(resp_fig, 380).update_layout(showlegend=True, title_text="Отклик на движение Binance")
resp_fig.update_xaxes(title_text="горизонт, мс")
resp_fig.update_yaxes(title_text="сдвиг mid в сторону события, bps")
resp_fig.show()
response.round(3)

ОТВЕТ A: Lighter впитывает 50% движения Binance за 187 мс, 90% — за 3068 мс


,binance_bps,lighter_bps,absorbed
0ms,1.745,0.000,0.000
25ms,2.285,0.211,0.092
50ms,2.360,0.490,0.208
100ms,2.496,0.882,0.353
200ms,2.567,1.341,0.523
350ms,2.624,1.635,0.623
500ms,2.659,1.794,0.675
1000ms,2.709,2.035,0.751
2000ms,2.743,2.356,0.859
5000ms,2.810,2.738,0.975


##### B. Время до первой реакции Lighter

In [20]:
REACTION_TICKS = 1     # реакция = mid Lighter сдвинулся в сторону события на ≥ стольких тиков
MAX_WAIT_MS = 5000     # дольше не ждём: событие считаем «без реакции» (цензурировано)


def first_reaction_ms(event_ts, event_sign, ts, mid, min_move, max_wait_us):
    """Для каждого события: мс до первого сдвига mid Lighter в сторону события на ≥ min_move.
    NaN = реакции не было в окне (цензурировано) либо не было опорного апдейта до события."""
    out = np.full(len(event_ts), np.nan)
    ref = np.searchsorted(ts, event_ts, side="left") - 1    # последний апдейт ДО события
    start = np.searchsorted(ts, event_ts, side="right")     # первый апдейт строго ПОСЛЕ
    for k in range(len(event_ts)):
        if ref[k] < 0:
            continue
        base = mid[ref[k]]
        deadline = event_ts[k] + max_wait_us
        for j in range(start[k], len(ts)):
            if ts[j] > deadline:
                break
            if event_sign[k] * (mid[j] - base) >= min_move:
                out[k] = (ts[j] - event_ts[k]) / 1000
                break
    return out


reaction_ms = first_reaction_ms(event_ts, event_sign, lighter_ts, lighter_mid,
                                REACTION_TICKS * TICK, MAX_WAIT_MS * 1000)
reacted = ~np.isnan(reaction_ms)
print(f"ОТВЕТ B: Lighter впервые двигается в сторону события через "
      f"p50={np.nanpercentile(reaction_ms, 50):.0f} мс, p90={np.nanpercentile(reaction_ms, 90):.0f} мс")
print(f"         реакция была у {reacted.sum():,} из {len(reaction_ms):,} событий "
      f"({reacted.mean() * 100:.1f}%); остальные — без реакции за {MAX_WAIT_MS} мс")

ОТВЕТ B: Lighter впервые двигается в сторону события через p50=169 мс, p90=2118 мс
         реакция была у 35,155 из 46,441 событий (75.7%); остальные — без реакции за 5000 мс


##### C. Кросс-корреляция доходностей: на каком лаге максимум

In [21]:
GRID_MS = 50           # шаг общей сетки
MAX_LAG_BINS = 40      # ищем лаг в диапазоне ±2 с


def grid_prices(ts, mid, grid_ts):
    """Последняя известная цена на момент каждого узла сетки (as-of)."""
    idx = np.searchsorted(ts, grid_ts, side="right") - 1
    return np.where(idx >= 0, mid[np.clip(idx, 0, None)], np.nan)


def corr_at_lag(a, b, k):
    """corr(a[t], b[t-k]); k > 0 → a отстаёт от b на k бинов."""
    if k > 0:
        x, y = a[k:], b[:-k]
    elif k < 0:
        x, y = a[:k], b[-k:]
    else:
        x, y = a, b
    return float(np.corrcoef(x, y)[0, 1])


grid_ts = np.arange(max(binance_ts[0], lighter_ts[0]),
                    min(binance_ts[-1], lighter_ts[-1]), GRID_MS * 1000)
px_binance = grid_prices(binance_ts, binance_mid, grid_ts)
px_lighter = grid_prices(lighter_ts, lighter_mid, grid_ts)
ret_binance = np.diff(px_binance) / px_binance[:-1] * 1e4
ret_lighter = np.diff(px_lighter) / px_lighter[:-1] * 1e4

lags = np.arange(-MAX_LAG_BINS, MAX_LAG_BINS + 1)
corr = np.array([corr_at_lag(ret_lighter, ret_binance, int(k)) for k in lags])
best_lag = int(lags[np.nanargmax(corr)])
print(f"ОТВЕТ C: максимум кросс-корреляции на лаге {best_lag} бинов = {best_lag * GRID_MS} мс "
      f"(corr={np.nanmax(corr):.3f}), на лаге 0 corr={corr[lags == 0][0]:.3f}")
print(f"         положительный лаг = Lighter отстаёт от Binance")

corr_fig = go.Figure(go.Scatter(x=lags * GRID_MS, y=corr, mode="lines+markers",
                                line=dict(color="#4C78A8")))
corr_fig.add_vline(x=best_lag * GRID_MS, line=dict(color="#E45756", dash="dot", width=1))
corr_fig.add_vline(x=0, line=dict(color="#999999", width=1))
style_figure(corr_fig, 380).update_layout(title_text="corr(Δmid Lighter(t), Δmid Binance(t−lag))")
corr_fig.update_xaxes(title_text="лаг, мс  (>0 = Lighter отстаёт)")
corr_fig.update_yaxes(title_text="корреляция")
corr_fig.show()

ОТВЕТ C: максимум кросс-корреляции на лаге 1 бинов = 50 мс (corr=0.187), на лаге 0 corr=0.100
         положительный лаг = Lighter отстаёт от Binance


### 2. Order Book

#### Какой средний spread в bps и в тиках?

In [22]:
# Спред из top-of-book книги. Валидность: обе стороны есть и не скрещены.
best_ask = book["ask_px_1"].to_numpy()
best_bid = book["bid_px_1"].to_numpy()
valid_book = (best_ask > 0) & (best_bid > 0) & (best_ask >= best_bid)

mid = (best_ask[valid_book] + best_bid[valid_book]) / 2
abs_spread = best_ask[valid_book] - best_bid[valid_book]
ref_price = float(np.median(mid))

spread = pd.DataFrame({
    "recv_ts": book["recv_ts"].to_numpy()[valid_book],
    "ticks": abs_spread / TICK,
    "bps": abs_spread / mid * 1e4,
})

print(f"valid snapshots: {valid_book.sum():,} / {len(book):,} (dropped {(~valid_book).sum()}) | "
      f"ref price {ref_price:.4g} | 1 тик = {TICK / ref_price * 1e4:.3f} bps")
spread.head()

valid snapshots: 1,882,508 / 1,882,508 (dropped 0) | ref price 0.6182 | 1 тик = 0.162 bps


,recv_ts,ticks,bps
0,1783987200090027,78.0,14.216455
1,1783987200213904,85.0,15.491302
2,1783987200265156,85.0,15.491302
3,1783987200281873,85.0,15.491302
4,1783987200349793,85.0,15.491302


In [23]:
# Распределение спреда: тики слева, bps справа. Линии p50 (сплошная) и p90 (пунктир).
hist = make_subplots(rows=1, cols=2,
                     subplot_titles=("Spread distribution, ticks", "Spread distribution, bps"))
for unit, col in (("ticks", 1), ("bps", 2)):
    hist.add_trace(go.Histogram(x=spread[unit], nbinsx=80, marker_color="#4C78A8"), 1, col)
    for level, dash in ((50, "solid"), (90, "dot")):
        hist.add_vline(x=float(np.percentile(spread[unit], level)),
                       line=dict(color="#E45756", dash=dash, width=1), row=1, col=col)
    hist.update_xaxes(range=[0, float(np.percentile(spread[unit], 99))], title_text=unit, row=1, col=col)

style_figure(hist, 380).show()

In [24]:
def spread_stats(series):
    """p50 / p90 / p99 / mean одной серии спреда."""
    return {
        "p50": round(float(np.percentile(series, 50)), 3),
        "p90": round(float(np.percentile(series, 90)), 3),
        "p99": round(float(np.percentile(series, 99)), 3),
        "mean": round(float(series.mean()), 3),
    }


spread_table = pd.DataFrame([spread_stats(spread["ticks"]), spread_stats(spread["bps"])],
                            index=["ticks", "bps"])
print("Спред на Lighter (по снапшотам книги)")
spread_table

Спред на Lighter (по снапшотам книги)


,p50,p90,p99,mean
ticks,89.000,126.000,167.000,90.435
bps,14.613,20.184,26.657,14.717


#### Насколько OrderBook плотный?

«Плотность» — три разные вещи: **сколько $ на уровнях** (A, B), **насколько плотно заняты тики**
(C) и **докуда книга нам вообще видна** (E). Считаем от **mid** (экономика: маркет-ордер сначала
платит спред) и от **тача** (структура конкуренции за нами).

In [25]:
# Плотность книги на неделе: полные 30-уровневые матрицы (~12 ГБ) НЕ материализуем. Потоковый проход
# по файлам (один резидентно, ~22 МБ), на выходе — ТОЧНЫЕ скаляры на снапшот (кумулятивный $ в полосах,
# occupancy, горизонт, L1) + гистограммы для профиля/зазоров (медианы по уровням не сводятся к скаляру).
LEVELS = 30
LEVEL_COLUMNS = ["recv_ts"] + [f"{s}_{f}_{k}" for s in ("ask", "bid")
                               for f in ("px", "sz") for k in range(1, LEVELS + 1)]
BPS_LIMITS = [2, 5, 10, 20, 25, 50]        # супермножество полос от mid (dens-b ∪ имбаланс ∪ dens-e)
TICK_LIMITS = [0, 10, 25, 50, 100, 200]    # полосы от тача (dens-b)
OCCUPANCY_TICKS = 100                      # окно occupancy (dens-c)


def tw_percentiles(values, times, levels):
    """Перцентили, взвешенные по ВРЕМЕНИ жизни снапшота, а не по их числу."""
    dwell = np.clip(np.diff(times, append=times[-1]), 0, None)
    order = np.argsort(values)
    cw = np.cumsum(dwell[order])
    return values[order][np.searchsorted(cw, np.array(levels) / 100 * cw[-1])]


class LevelHistogram:
    """Гистограммы с фиксированными бинами по каждому уровню; перцентиль читается из счётчиков.
    Заменяет удержание всех значений в памяти — точность равна ширине бина."""

    def __init__(self, lo, hi, width, n_levels=LEVELS):
        self.edges = np.arange(lo, hi + width, width)
        self.width, self.lo, self.hi = width, lo, hi
        self.counts = np.zeros((n_levels, len(self.edges) - 1))

    def add(self, values):                       # values [n, n_levels] (или [n, 1]); NaN пропускаем
        for k in range(values.shape[1]):
            col = values[:, k]
            col = np.clip(col[np.isfinite(col)], self.lo, self.hi - self.width / 2)
            if len(col):
                self.counts[k] += np.histogram(col, bins=self.edges)[0]

    def percentile(self, q):
        out = np.full(self.counts.shape[0], np.nan)
        for k, row in enumerate(self.counts):
            cw = np.cumsum(row)
            if cw[-1] > 0:
                out[k] = self.edges[min(int(np.searchsorted(cw, q / 100 * cw[-1])), len(row) - 1)] + self.width / 2
        return out


# ── аккумуляторы: скаляры на снапшот копим списком по файлам, затем конкат ──
series = {name: [] for name in (
    "book_ts", "mid", "l1_bid_usd", "l1_ask_usd", "l1_bid_px", "l1_ask_px", "l1_bid_sz", "l1_ask_sz",
    "total_bid_usd", "total_ask_usd", "ask_horizon_bps", "bid_horizon_bps",
    "ask_levels_filled", "bid_levels_filled", "occupancy_levels_within")}
cum_bid_bps = {lim: [] for lim in BPS_LIMITS};   cum_ask_bps = {lim: [] for lim in BPS_LIMITS}
cum_bid_ticks = {lim: [] for lim in TICK_LIMITS}; cum_ask_ticks = {lim: [] for lim in TICK_LIMITS}

hist_bid_usd = LevelHistogram(0, 100_000, 5);  hist_ask_usd = LevelHistogram(0, 100_000, 5)
hist_bid_dist = LevelHistogram(0, 3_000, 1);   hist_ask_dist = LevelHistogram(0, 3_000, 1)
hist_gaps = LevelHistogram(0, 5_000, 1, n_levels=1)
filled_bid_by_level = np.zeros(LEVELS);  filled_ask_by_level = np.zeros(LEVELS)
n_snapshots = 0

for chunk in LobReaderRaw(lob_paths, columns=LEVEL_COLUMNS):
    a_px = chunk[[f"ask_px_{k}" for k in range(1, LEVELS + 1)]].to_numpy()
    a_sz = chunk[[f"ask_sz_{k}" for k in range(1, LEVELS + 1)]].to_numpy()
    b_px = chunk[[f"bid_px_{k}" for k in range(1, LEVELS + 1)]].to_numpy()
    b_sz = chunk[[f"bid_sz_{k}" for k in range(1, LEVELS + 1)]].to_numpy()
    valid = (a_px[:, 0] > 0) & (b_px[:, 0] > 0) & (a_px[:, 0] >= b_px[:, 0])
    if not valid.any():
        continue
    a_px, a_sz, b_px, b_sz = a_px[valid], a_sz[valid], b_px[valid], b_sz[valid]
    m = (a_px[:, 0] + b_px[:, 0]) / 2
    mcol = m[:, None]
    a_filled, b_filled = a_px > 0, b_px > 0
    a_usd, b_usd = a_px * a_sz, b_px * b_sz
    a_from_mid = np.where(a_filled, (a_px - mcol) / mcol * 1e4, np.nan)
    b_from_mid = np.where(b_filled, (mcol - b_px) / mcol * 1e4, np.nan)
    a_from_touch = np.where(a_filled, (a_px - a_px[:, :1]) / TICK, np.nan)
    b_from_touch = np.where(b_filled, (b_px[:, :1] - b_px) / TICK, np.nan)

    series["book_ts"].append(chunk["recv_ts"].to_numpy()[valid]); series["mid"].append(m)
    series["l1_bid_usd"].append(b_usd[:, 0]); series["l1_ask_usd"].append(a_usd[:, 0])
    series["l1_bid_px"].append(b_px[:, 0]); series["l1_ask_px"].append(a_px[:, 0])
    series["l1_bid_sz"].append(b_sz[:, 0]); series["l1_ask_sz"].append(a_sz[:, 0])
    series["total_bid_usd"].append(b_usd.sum(1)); series["total_ask_usd"].append(a_usd.sum(1))
    series["ask_horizon_bps"].append(np.nanmax(a_from_mid, axis=1))
    series["bid_horizon_bps"].append(np.nanmax(b_from_mid, axis=1))
    series["ask_levels_filled"].append(a_filled.sum(1)); series["bid_levels_filled"].append(b_filled.sum(1))
    series["occupancy_levels_within"].append(
        (((a_from_touch <= OCCUPANCY_TICKS) & a_filled).sum(1)
         + ((b_from_touch <= OCCUPANCY_TICKS) & b_filled).sum(1)) / 2)
    for lim in BPS_LIMITS:
        cum_bid_bps[lim].append(np.where(b_from_mid <= lim, b_usd, 0.0).sum(1))
        cum_ask_bps[lim].append(np.where(a_from_mid <= lim, a_usd, 0.0).sum(1))
    for lim in TICK_LIMITS:
        cum_bid_ticks[lim].append(np.where(b_from_touch <= lim, b_usd, 0.0).sum(1))
        cum_ask_ticks[lim].append(np.where(a_from_touch <= lim, a_usd, 0.0).sum(1))

    hist_bid_usd.add(np.where(b_filled, b_usd, np.nan)); hist_ask_usd.add(np.where(a_filled, a_usd, np.nan))
    hist_bid_dist.add(b_from_touch); hist_ask_dist.add(a_from_touch)
    a_gaps = np.diff(np.where(a_filled, a_px, np.nan), axis=1) / TICK
    b_gaps = -np.diff(np.where(b_filled, b_px, np.nan), axis=1) / TICK   # цены бида убывают
    hist_gaps.add(np.concatenate([a_gaps[~np.isnan(a_gaps)], b_gaps[~np.isnan(b_gaps)]])[:, None])
    filled_ask_by_level += a_filled.sum(0); filled_bid_by_level += b_filled.sum(0)
    n_snapshots += len(m)

series = {name: np.concatenate(parts) for name, parts in series.items()}
cum_bid_bps = {lim: np.concatenate(v) for lim, v in cum_bid_bps.items()}
cum_ask_bps = {lim: np.concatenate(v) for lim, v in cum_ask_bps.items()}
cum_bid_ticks = {lim: np.concatenate(v) for lim, v in cum_bid_ticks.items()}
cum_ask_ticks = {lim: np.concatenate(v) for lim, v in cum_ask_ticks.items()}
book_ts, mid = series["book_ts"], series["mid"]

print(f"снапшотов: {n_snapshots:,} | 1 тик = {TICK / ref_price * 1e4:.3f} bps | "
      f"спред p50 = {np.percentile(spread['ticks'], 50):.0f} тиков")

снапшотов: 1,882,508 | 1 тик = 0.162 bps | спред p50 = 89 тиков


##### A. Профиль глубины по уровням
**Зачем:** увидеть, пылевой ли L1 или стена, и где сидят «настоящие» деньги — это те, с кем мы делим очередь.

In [26]:
# Профиль по уровням: медианы читаются из накопленных гистограмм (точность = ширина бина: $5 / 1 тик).
profile = pd.DataFrame({
    "bid_usd_p50": hist_bid_usd.percentile(50),
    "ask_usd_p50": hist_ask_usd.percentile(50),
    "bid_dist_ticks_p50": hist_bid_dist.percentile(50),
    "ask_dist_ticks_p50": hist_ask_dist.percentile(50),
    "filled_share": (filled_ask_by_level + filled_bid_by_level) / (2 * n_snapshots),
}, index=[f"L{k}" for k in range(1, LEVELS + 1)])

fattest = profile[["bid_usd_p50", "ask_usd_p50"]].mean(axis=1).idxmax()
print(f"ОТВЕТ A: на туше (L1) стоит ${profile.loc['L1', 'bid_usd_p50']:,.0f} (bid) / "
      f"${profile.loc['L1', 'ask_usd_p50']:,.0f} (ask); самый жирный уровень — {fattest}")

levels_x = np.arange(1, LEVELS + 1)
prof_fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                         subplot_titles=("Медиана $ на уровне", "Медиана расстояния от тача, тики"))
prof_fig.add_trace(go.Scatter(x=levels_x, y=profile["bid_usd_p50"], name="bid",
                              line=dict(color="#4C78A8")), 1, 1)
prof_fig.add_trace(go.Scatter(x=levels_x, y=profile["ask_usd_p50"], name="ask",
                              line=dict(color="#E45756")), 1, 1)
prof_fig.add_trace(go.Scatter(x=levels_x, y=profile["bid_dist_ticks_p50"], showlegend=False,
                              line=dict(color="#4C78A8", dash="dot")), 2, 1)
prof_fig.add_trace(go.Scatter(x=levels_x, y=profile["ask_dist_ticks_p50"], showlegend=False,
                              line=dict(color="#E45756", dash="dot")), 2, 1)
style_figure(prof_fig, 520).update_layout(showlegend=True)
prof_fig.update_xaxes(title_text="уровень", row=2, col=1)
prof_fig.update_yaxes(title_text="$", row=1, col=1)
prof_fig.update_yaxes(title_text="тики от тача", row=2, col=1)
prof_fig.show()
profile.round(1)

ОТВЕТ A: на туше (L1) стоит $1,028 (bid) / $1,112 (ask); самый жирный уровень — L18


,bid_usd_p50,ask_usd_p50,bid_dist_ticks_p50,ask_dist_ticks_p50,filled_share
L1,1027.5,1112.5,0.5,0.5,1.0
L2,8072.5,10012.5,22.5,7.5,1.0
L3,9652.5,10042.5,39.5,21.5,1.0
L4,9697.5,10007.5,53.5,31.5,1.0
L5,9622.5,5502.5,70.5,46.5,1.0
L6,5742.5,3282.5,103.5,59.5,1.0
L7,10077.5,3282.5,183.5,73.5,1.0
L8,10002.5,4097.5,281.5,86.5,1.0
L9,10027.5,4102.5,539.5,103.5,1.0
L10,16682.5,1002.5,644.5,178.5,1.0


##### B. Кумулятивная глубина: от mid (bps) и от тача (тики)
**Зачем:** «сколько $ лежит в пределах X от цены» — это и ликвидность, которая нас absorbs при выходе,
и объём, который надо пройти, чтобы нас снять. Перцентили **взвешены по времени**, а не по снапшотам.

In [27]:
DIST_BPS = [2, 5, 10, 20, 50]            # ⊆ BPS_LIMITS (посчитаны потоково в dens-prep)
DIST_TICKS = [0, 10, 25, 50, 100, 200]   # = TICK_LIMITS


def depth_table(cum_bid, cum_ask, limits, label):
    rows = []
    for limit in limits:
        b = tw_percentiles(cum_bid[limit], book_ts, [25, 50, 75])
        a = tw_percentiles(cum_ask[limit], book_ts, [25, 50, 75])
        rows.append({label: limit, "bid_p25": b[0], "bid_p50": b[1], "bid_p75": b[2],
                     "ask_p25": a[0], "ask_p50": a[1], "ask_p75": a[2]})
    return pd.DataFrame(rows).set_index(label).round(0)


depth_from_mid = depth_table(cum_bid_bps, cum_ask_bps, DIST_BPS, "bps от mid")
depth_from_touch = depth_table(cum_bid_ticks, cum_ask_ticks, DIST_TICKS, "тиков от тача")

print(f"ОТВЕТ B: внутри 10 bps от mid стоит ${depth_from_mid.loc[10, 'bid_p50']:,.0f} (bid) / "
      f"${depth_from_mid.loc[10, 'ask_p50']:,.0f} (ask) — медиана по времени")
print(f"         на самом туше (0 тиков): ${depth_from_touch.loc[0, 'bid_p50']:,.0f} / "
      f"${depth_from_touch.loc[0, 'ask_p50']:,.0f}")
display(depth_from_mid)
depth_from_touch

ОТВЕТ B: внутри 10 bps от mid стоит $1,122 (bid) / $18,836 (ask) — медиана по времени
         на самом туше (0 тиков): $1,033 / $8,093


,bid_p25,bid_p50,bid_p75,ask_p25,ask_p50,ask_p75
bps от mid,,,,,,
2,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0
10,961.0,1122.0,9087.0,8982.0,18836.0,29216.0
20,28982.0,36000.0,43270.0,46782.0,50791.0,54485.0
50,44803.0,45758.0,46515.0,60075.0,61013.0,61688.0


,bid_p25,bid_p50,bid_p75,ask_p25,ask_p50,ask_p75
тиков от тача,,,,,,
0,927.0,1033.0,9005.0,977.0,8093.0,9083.0
10,956.0,1079.0,9085.0,1158.0,10168.0,19270.0
25,998.0,1322.0,9208.0,10077.0,24479.0,34915.0
50,1993.0,10861.0,29228.0,34949.0,40145.0,46658.0
100,35766.0,43951.0,44929.0,51578.0,55650.0,59969.0
200,44464.0,45327.0,46156.0,59730.0,60618.0,61335.0


##### C. Разреженность ценовой сетки
**Зачем:** если книга дырявая, мы можем встать в **пустой ценовой слот и быть там одни** (очереди нет).
И наоборот: большие зазоры = маркет-ордер «прыгает» далеко.

In [28]:
# Зазоры между соседними уровнями — из накопленной гистограммы (точность 1 тик). Occupancy — точная серия.
gap_p50 = hist_gaps.percentile(50)[0]
gap_p90 = hist_gaps.percentile(90)[0]
gap_p99 = hist_gaps.percentile(99)[0]
occupancy_pct = series["occupancy_levels_within"] / OCCUPANCY_TICKS * 100

print(f"ОТВЕТ C: соседние уровни стоят через {gap_p50:.0f} тиков (медиана), "
      f"p90 = {gap_p90:.0f} тиков")
print(f"         в первых {OCCUPANCY_TICKS} тиках за тачем занято {np.median(occupancy_pct):.1f}% "
      f"ценовых слотов ({np.median(series['occupancy_levels_within']):.0f} уровней из {OCCUPANCY_TICKS}) "
      f"→ книга дырявая, места встать одному много")

gap_centers = hist_gaps.edges[:-1] + hist_gaps.width / 2
gap_counts = hist_gaps.counts[0]
shown = gap_centers <= gap_p99
gap_fig = go.Figure(go.Bar(x=gap_centers[shown], y=gap_counts[shown], marker_color="#4C78A8"))
gap_fig.add_vline(x=float(gap_p50), line=dict(color="#E45756", width=1))
style_figure(gap_fig, 360).update_layout(title_text="Зазор между соседними уровнями, тики")
gap_fig.update_xaxes(title_text="тики")
gap_fig.show()

ОТВЕТ C: соседние уровни стоят через 62 тиков (медиана), p90 = 620 тиков
         в первых 100 тиках за тачем занято 7.0% ценовых слотов (7 уровней из 100) → книга дырявая, места встать одному много


##### E. Покрытие 30 уровней и асимметрия сторон
**Зачем:** это **гард для B**. Мы видим только 30 уровней — если они достают, скажем, до 30 bps,
то любые оценки глубины дальше этого горизонта просто **обрезаны** и им нельзя верить.

In [29]:
ask_horizon_bps = series["ask_horizon_bps"]   # докуда достаёт последний видимый уровень
bid_horizon_bps = series["bid_horizon_bps"]

REF_BPS = 10   # ∈ BPS_LIMITS
asymmetry = np.divide(cum_bid_bps[REF_BPS], cum_ask_bps[REF_BPS],
                      out=np.full_like(cum_bid_bps[REF_BPS], np.nan), where=cum_ask_bps[REF_BPS] > 0)

coverage = pd.DataFrame({
    "levels_filled_p50": [np.median(series["bid_levels_filled"]), np.median(series["ask_levels_filled"])],
    "levels_filled_p10": [np.percentile(series["bid_levels_filled"], 10),
                          np.percentile(series["ask_levels_filled"], 10)],
    "horizon_bps_p50": [np.nanmedian(bid_horizon_bps), np.nanmedian(ask_horizon_bps)],
    "horizon_bps_p10": [np.nanpercentile(bid_horizon_bps, 10), np.nanpercentile(ask_horizon_bps, 10)],
}, index=["bid", "ask"])

visible_bps = np.nanmedian(np.minimum(ask_horizon_bps, bid_horizon_bps))
print(f"ОТВЕТ E: книга видна до {visible_bps:.1f} bps от mid (медиана) — ГЛУБЖЕ ОЦЕНКИ B НЕДОСТОВЕРНЫ")
print(f"         асимметрия bid/ask внутри {REF_BPS} bps: p50 = {np.nanmedian(asymmetry):.2f} "
      f"(1.0 = стороны симметричны)")
coverage.round(1)

ОТВЕТ E: книга видна до 613.9 bps от mid (медиана) — ГЛУБЖЕ ОЦЕНКИ B НЕДОСТОВЕРНЫ
         асимметрия bid/ask внутри 10 bps: p50 = 0.15 (1.0 = стороны симметричны)


,levels_filled_p50,levels_filled_p10,horizon_bps_p50,horizon_bps_p10
bid,30.0,30.0,1573.4,902.6
ask,30.0,30.0,626.3,370.0


### 3. Имбаланс

#### Как выглядит имбаланс?

Имбаланс книги в долларах: `I = (bid − ask) / (bid + ask)` ∈ [−1, +1], `I > 0` = перевес бида.

Считаем сразу на нескольких глубинах — на TAO выбор глубины решает всё:

- **touch** — только L1. Классика из литературы, но тач здесь пыль (~$1.5k bid / $250 ask),
  поэтому смотрим не столько на значение, сколько на то, **насколько он вырожден**
  (доля времени `|I| > 0.9`).
- **5 / 10 / 25 / 50 bps** — сумма $ в полосе от mid. Единственное честное сравнение сторон:
  в пункте A видно, что `L5` на биде стоит в 71 тике от тача, а на аске — в 92, так что top-K
  сравнивает разные расстояния. Полоса в bps одинакова для обеих сторон.
- **all30** — вся видимая книга, референс.

Распределение взвешиваем **по времени жизни снапшота**, а не по их числу: книга обновляется
рывками, иначе бурные секунды перевесят спокойные.

In [30]:
# Имбаланс книги: $ на обеих сторонах на ОДИНАКОВОМ расстоянии от mid (серии посчитаны потоково).
IMBALANCE_BANDS_BPS = [5, 10, 25, 50]   # ⊆ BPS_LIMITS
IMBALANCE_SHOWN = ["touch", "10bps", "50bps"]
IMBALANCE_COLORS = {"touch": "#c0504d", "10bps": "#4c78a8", "50bps": "#59a14f"}


def imbalance(bid_notional, ask_notional):
    """NaN, если полоса не покрыла ни одного уровня ни с одной стороны."""
    total = bid_notional + ask_notional
    return np.divide(bid_notional - ask_notional, total,
                     out=np.full_like(total, np.nan), where=total > 0)


imbalances = {"touch": imbalance(series["l1_bid_usd"], series["l1_ask_usd"])}
for band in IMBALANCE_BANDS_BPS:
    imbalances[f"{band}bps"] = imbalance(cum_bid_bps[band], cum_ask_bps[band])
imbalances["all30"] = imbalance(series["total_bid_usd"], series["total_ask_usd"])

book_time = pd.to_datetime(book_ts, unit="us")
dwell_us = np.clip(np.diff(book_ts, append=book_ts[-1]), 0, None)

undefined = {name: float(np.isnan(s).mean()) for name, s in imbalances.items()}
print(f"снапшотов: {len(book_ts):,} | глубины: {', '.join(imbalances)}")
print("I не определён (полоса пуста): " +
      (", ".join(f"{name} {share:.1%}" for name, share in undefined.items() if share > 0) or "нигде"))

снапшотов: 1,882,508 | глубины: touch, 5bps, 10bps, 25bps, 50bps, all30
I не определён (полоса пуста): 5bps 86.2%, 10bps 10.6%, 25bps 0.1%, 50bps 0.0%


In [31]:
SMOOTH_WINDOW_S = 2


def smooth_over_time(series, window_s):
    """Скользящая медиана по ВРЕМЕННОМУ окну — снапшоты приходят неравномерно."""
    return pd.Series(series, index=book_time).rolling(f"{window_s}s").median().to_numpy()


fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.28, 0.72],
                    subplot_titles=("mid", f"имбаланс книги: сырой (бледный) и медиана по {SMOOTH_WINDOW_S} с"))
fig.add_trace(go.Scattergl(x=book_time, y=mid, mode="lines", showlegend=False,
                           line=dict(color="#666", width=1)), row=1, col=1)

for name in IMBALANCE_SHOWN:
    color = IMBALANCE_COLORS[name]
    fig.add_trace(go.Scattergl(x=book_time, y=imbalances[name], mode="lines", opacity=0.18,
                               line=dict(color=color, width=0.5), showlegend=False, hoverinfo="skip"),
                  row=2, col=1)
    fig.add_trace(go.Scattergl(x=book_time, y=smooth_over_time(imbalances[name], SMOOTH_WINDOW_S),
                               mode="lines", name=name, line=dict(color=color, width=1.3)), row=2, col=1)

for level, dash in ((0, "solid"), (0.5, "dot"), (-0.5, "dot")):
    fig.add_hline(y=level, row=2, col=1, line_width=1, line_dash=dash, line_color="#999")

fig.update_yaxes(title_text="цена", row=1, col=1)
fig.update_yaxes(title_text="I", range=[-1.05, 1.05], row=2, col=1)
fig.update_xaxes(title_text="время (UTC)", row=2, col=1)
style_figure(fig, 640).update_layout(showlegend=True, legend=dict(orientation="h", y=1.05, x=0)).show()

In [32]:
PERCENTILE_LEVELS = [5, 25, 50, 75, 95]


def imbalance_stats(series):
    """Все статистики взвешены по времени жизни снапшота, а не по их числу."""
    defined = ~np.isnan(series)
    values, times = series[defined], book_ts[defined]
    dwell = np.clip(np.diff(times, append=times[-1]), 0, None)
    p5, p25, p50, p75, p95 = tw_percentiles(values, times, PERCENTILE_LEVELS)
    return {
        "p5": p5, "p25": p25, "p50": p50, "p75": p75, "p95": p95,
        "mean": np.average(values, weights=dwell),
        "|I| p50": tw_percentiles(np.abs(values), times, [50])[0],
        "время |I|>0.5": dwell[np.abs(values) > 0.5].sum() / dwell.sum(),
        "время |I|>0.9": dwell[np.abs(values) > 0.9].sum() / dwell.sum(),
    }


imbalance_table = pd.DataFrame({name: imbalance_stats(series)
                                for name, series in imbalances.items()}).T

fig = go.Figure()
for name in IMBALANCE_SHOWN:
    weights = np.where(np.isnan(imbalances[name]), 0.0, dwell_us)
    fig.add_trace(go.Histogram(x=imbalances[name], y=weights / weights.sum(), histfunc="sum",
                               xbins=dict(start=-1, end=1, size=0.05), name=name,
                               marker_color=IMBALANCE_COLORS[name], opacity=0.55))
fig.update_layout(barmode="overlay", title="Распределение имбаланса, взвешенное по времени")
fig.update_xaxes(title_text="I = (bid − ask) / (bid + ask)")
fig.update_yaxes(title_text="доля времени")
style_figure(fig, 420).update_layout(showlegend=True, legend=dict(orientation="h", y=1.08, x=0)).show()

core, touch = imbalance_table.loc["10bps"], imbalance_table.loc["touch"]
print(f"ОТВЕТ: имбаланс книги ±10 bps — медиана по времени {core['p50']:+.2f}, "
      f"IQR [{core['p25']:+.2f}, {core['p75']:+.2f}], |I| > 0.5 в {core['время |I|>0.5']:.0%} времени")
print(f"       на туше (L1): |I| > 0.9 в {touch['время |I|>0.9']:.0%} времени — "
      f"классическое определение вырождено")
imbalance_table.round(3)

ОТВЕТ: имбаланс книги ±10 bps — медиана по времени -0.77, IQR [-0.92, -0.01], |I| > 0.5 в 74% времени
       на туше (L1): |I| > 0.9 в 8% времени — классическое определение вырождено


,p5,p25,p50,p75,p95,mean,|I| p50,время |I|>0.5,время |I|>0.9
touch,-0.931,-0.798,-0.051,0.037,0.817,-0.204,0.788,0.632,0.080
5bps,-0.974,-0.814,-0.483,0.039,0.958,-0.277,0.781,0.648,0.224
10bps,-0.963,-0.918,-0.771,-0.005,0.853,-0.428,0.818,0.736,0.354
25bps,-0.259,-0.157,-0.146,-0.128,-0.060,-0.147,0.146,0.006,0.001
50bps,-0.183,-0.152,-0.144,-0.136,-0.071,-0.142,0.144,0.002,0.000
all30,-0.091,-0.074,0.012,0.093,0.105,0.009,0.081,0.000,0.000


#### Предсказывает ли имбаланс книги будущий Δmid на 0.2 / 0.5 / 1 / 5 / 30 с?

**Сэмплинг.** Равномерная сетка 1 с — одна и та же для всех горизонтов, поэтому они сравнимы
(по снапшотам сэмплить нельзя: бурные секунды перевесят спокойные). Предиктор — состояние книги
на последнем снапшоте **≤ t**; таргет — `Δmid` на `[t, t+h]`. Окна не пересекаются, заглядывания
вперёд нет.

**Механика против альфы.** Тонкий аск сам по себе двигает mid вверх: один мелкий тейкер его снимает,
лучшая цена шагает, mid прыгает. Это не прогноз, а арифметика — и на 0.2 с она будет доминировать.
Поэтому таргет меряем **от двух якорей**:

- от `mid(t)` — полный отклик (механика + сюрприз);
- от **микроцены** `(bid·ask_sz + ask·bid_sz) / (bid_sz + ask_sz)` — то есть от mid, уже поправленного
  на имбаланс тача. Разница между двумя β — это то, что имбаланс знает **сверх** механики.

**CI — блочный бутстрап**, блок = 5 × самый длинный горизонт. При h > 1 с окна перекрываются,
и обычный t-stat завысил бы значимость в разы.

In [33]:
HORIZONS_S = [0.2, 0.5, 1, 5, 30]
SAMPLE_STEP_S = 1                      # сетка сэмплов: общая для всех горизонтов
PREDICTORS = ["touch", "10bps", "50bps"]
BOOTSTRAP_DRAWS = 300
BLOCK_S = 5 * max(HORIZONS_S)          # блок бутстрапа длиннее самого длинного горизонта

sample_ts = np.arange(book_ts[0], book_ts[-1] - max(HORIZONS_S) * 1e6,
                      SAMPLE_STEP_S * 1e6).astype(np.int64)
at = np.searchsorted(book_ts, sample_ts, side="right") - 1   # последний снапшот ≤ t: вперёд не заглядываем

# Микроцена — mid, уже поправленный на имбаланс тача: механически известная часть будущего движения.
touch_qty = series["l1_bid_sz"] + series["l1_ask_sz"]
microprice = np.divide(series["l1_bid_px"] * series["l1_ask_sz"] + series["l1_ask_px"] * series["l1_bid_sz"],
                       touch_qty, out=mid.copy(), where=touch_qty > 0)


def future_drift(anchor):
    """(mid(t+h) − anchor(t)) / mid(t) в bps; матрица [сэмплы × горизонты]."""
    drift = np.empty((len(sample_ts), len(HORIZONS_S)))
    for j, h in enumerate(HORIZONS_S):
        ahead = np.searchsorted(book_ts, sample_ts + int(h * 1e6), side="right") - 1
        drift[:, j] = (mid[ahead] - anchor[at]) / mid[at] * 1e4
    return drift


drift_from_mid = future_drift(mid)
drift_from_micro = future_drift(microprice)
predictors = {name: imbalances[name][at] for name in PREDICTORS}
half_spread_bps = float(np.percentile(spread["bps"], 50)) / 2

sampling = pd.DataFrame({
    "независимых окон": [int(len(sample_ts) * min(1, SAMPLE_STEP_S / h)) for h in HORIZONS_S],
    "доля Δmid = 0": [(drift_from_mid[:, j] == 0).mean() for j in range(len(HORIZONS_S))],
    "|Δmid| p50, bps": [np.median(np.abs(drift_from_mid[:, j])) for j in range(len(HORIZONS_S))],
    "|Δmid| p95, bps": [np.percentile(np.abs(drift_from_mid[:, j]), 95) for j in range(len(HORIZONS_S))],
}, index=pd.Index(HORIZONS_S, name="горизонт, с"))

undefined_predictor = {name: float(np.isnan(s).mean()) for name, s in predictors.items()}
print(f"сэмплов: {len(sample_ts):,} на сетке {SAMPLE_STEP_S} с | полуспред p50 = {half_spread_bps:.2f} bps | "
      f"блок бутстрапа = {BLOCK_S:.0f} с")
print("предиктор не определён: " +
      (", ".join(f"I_{n} {s:.1%}" for n, s in undefined_predictor.items() if s > 0) or "нигде"))
sampling.round(3)

сэмплов: 399,570 на сетке 1 с | полуспред p50 = 7.31 bps | блок бутстрапа = 150 с
предиктор не определён: I_10bps 6.2%, I_50bps 0.0%


,независимых окон,доля Δmid = 0,"|Δmid| p50, bps","|Δmid| p95, bps"
"горизонт, с",,,,
0.2,399570,0.898,0.000,0.407
0.5,399570,0.804,0.000,1.374
1.0,399570,0.727,0.000,2.817
5.0,79914,0.452,0.163,7.669
30.0,13319,0.095,3.789,18.474


##### A. Кривая отклика: средний Δmid по бакетам имбаланса

Квантильные бакеты (равное N в каждом), по оси x — среднее значение имбаланса внутри бакета.
**Амплитуда** = разница между верхним и нижним бакетом, в bps и в долях полуспреда.

In [34]:
N_BUCKETS = 5          # квантильные бакеты; на 24 ч данных можно поднять до 10
BLOCK_SAMPLES = int(BLOCK_S / SAMPLE_STEP_S)
HORIZON_RGB = ["76,120,168", "245,133,24", "84,162,75", "228,87,86", "114,183,178"]


def bucket_by_quantile(values, n_buckets):
    edges = np.quantile(values, np.linspace(0, 1, n_buckets + 1))
    return np.clip(np.searchsorted(edges[1:-1], values, side="right"), 0, n_buckets - 1)


def bucket_means(bucket, target, n_buckets):
    total = np.bincount(bucket, weights=target, minlength=n_buckets)
    count = np.bincount(bucket, minlength=n_buckets)
    return np.where(count > 0, total / np.maximum(count, 1), np.nan)


def block_indices(n, block, rng):
    """Блочный бутстрап: склеиваем случайные куски ПОДРЯД идущих наблюдений — так сохраняется
    автокорреляция, из-за которой перекрывающиеся окна завышают значимость."""
    starts = rng.integers(0, n - block + 1, size=int(np.ceil(n / block)))
    return (starts[:, None] + np.arange(block)).ravel()[:n]


def response_curve(x, targets, n_buckets, block, draws):
    """Средний Δmid в каждом бакете предиктора + CI. Возвращает (mean, lo, hi, x_центр бакета)."""
    defined = np.isfinite(x)
    x, targets = x[defined], targets[defined]
    bucket = bucket_by_quantile(x, n_buckets)
    mean = np.stack([bucket_means(bucket, targets[:, j], n_buckets)
                     for j in range(targets.shape[1])], axis=1)

    rng = np.random.default_rng(0)
    resampled = np.empty((draws, n_buckets, targets.shape[1]))
    for d in range(draws):
        take = block_indices(len(x), block, rng)
        for j in range(targets.shape[1]):
            resampled[d, :, j] = bucket_means(bucket[take], targets[take, j], n_buckets)
    lo, hi = np.nanpercentile(resampled, [2.5, 97.5], axis=0)
    return mean, lo, hi, bucket_means(bucket, x, n_buckets)


fig = make_subplots(rows=1, cols=len(PREDICTORS), shared_yaxes=True, horizontal_spacing=0.03,
                    subplot_titles=[f"I_{name}" for name in PREDICTORS])
response = {}
for col, name in enumerate(PREDICTORS, start=1):
    mean, lo, hi, bucket_x = response_curve(predictors[name], drift_from_mid,
                                            N_BUCKETS, BLOCK_SAMPLES, BOOTSTRAP_DRAWS)
    response[name] = mean
    for j, h in enumerate(HORIZONS_S):
        rgb = HORIZON_RGB[j]
        fig.add_trace(go.Scatter(x=np.r_[bucket_x, bucket_x[::-1]], y=np.r_[hi[:, j], lo[::-1, j]],
                                 fill="toself", fillcolor=f"rgba({rgb},0.10)", line_width=0,
                                 hoverinfo="skip", showlegend=False), row=1, col=col)
        fig.add_trace(go.Scatter(x=bucket_x, y=mean[:, j], mode="lines+markers", name=f"{h} с",
                                 line=dict(color=f"rgb({rgb})", width=1.6), marker_size=5,
                                 showlegend=(col == 1)), row=1, col=col)

fig.add_hline(y=0, line_width=1, line_color="#bbb")
fig.update_yaxes(title_text="средний Δmid, bps", row=1, col=1)
fig.update_xaxes(title_text="имбаланс (среднее в бакете)")
style_figure(fig, 430).update_layout(
    title=f"Отклик mid на имбаланс книги — для масштаба: полуспред = {half_spread_bps:.2f} bps",
    showlegend=True, legend=dict(orientation="h", y=1.12, x=0), margin_t=95).show()

amplitude = pd.DataFrame({f"{h} с": [response[name][-1, j] - response[name][0, j] for name in PREDICTORS]
                          for j, h in enumerate(HORIZONS_S)}, index=[f"I_{name}" for name in PREDICTORS])
widest = amplitude.abs().max(axis=1).idxmax()
widest_h = amplitude.loc[widest].abs().idxmax()
print(f"ОТВЕТ A: самый широкий отклик — {widest} на {widest_h}: между крайними бакетами "
      f"{amplitude.loc[widest, widest_h]:+.3f} bps = "
      f"{abs(amplitude.loc[widest, widest_h]) / half_spread_bps:.0%} полуспреда")
print("Амплитуда (верхний бакет − нижний), bps:")
display(amplitude.round(3))
print("Она же в долях полуспреда:")
(amplitude / half_spread_bps).round(2)

ОТВЕТ A: самый широкий отклик — I_10bps на 30 с: между крайними бакетами +2.032 bps = 28% полуспреда
Амплитуда (верхний бакет − нижний), bps:


,0.2 с,0.5 с,1 с,5 с,30 с
I_touch,0.077,0.157,0.259,0.786,1.614
I_10bps,0.099,0.221,0.366,1.027,2.032
I_50bps,0.022,0.045,0.071,0.157,0.402


Она же в долях полуспреда:


,0.2 с,0.5 с,1 с,5 с,30 с
I_touch,0.01,0.02,0.04,0.11,0.22
I_10bps,0.01,0.03,0.05,0.14,0.28
I_50bps,0.00,0.01,0.01,0.02,0.06


##### B. Регрессия: β в bps на +1σ имбаланса

Одно число на клетку, сравнимое между предикторами и горизонтами. Рядом — `R²` (сколько дисперсии
`Δmid` объяснено вообще) и **β от микроцены**: та же регрессия, но механическая часть будущего
движения уже вычтена из таргета.

In [35]:
def betas(x, targets):
    """β в bps на +1σ предиктора и R², векторно по горизонтам."""
    standardized = (x - x.mean()) / x.std()
    beta = standardized @ (targets - targets.mean(axis=0)) / len(standardized)
    return beta, (beta / targets.std(axis=0)) ** 2


def beta_ci(x, targets, block, draws):
    rng = np.random.default_rng(1)
    resampled = np.empty((draws, targets.shape[1]))
    for d in range(draws):
        take = block_indices(len(x), block, rng)
        resampled[d] = betas(x[take], targets[take])[0]
    return np.percentile(resampled, [2.5, 97.5], axis=0)


rows = []
for name in PREDICTORS:
    defined = np.isfinite(predictors[name])
    x = predictors[name][defined]
    beta_mid, r2_mid = betas(x, drift_from_mid[defined])
    beta_micro, _ = betas(x, drift_from_micro[defined])
    lo, hi = beta_ci(x, drift_from_mid[defined], BLOCK_SAMPLES, BOOTSTRAP_DRAWS)
    for j, h in enumerate(HORIZONS_S):
        rows.append({"предиктор": f"I_{name}", "h, с": h,
                     "β, bps/σ": beta_mid[j], "CI_lo": lo[j], "CI_hi": hi[j],
                     "β/полуспред": beta_mid[j] / half_spread_bps, "R²": r2_mid[j],
                     "β от микроцены": beta_micro[j]})

regression = pd.DataFrame(rows)
regression["CI не содержит 0"] = np.sign(regression["CI_lo"]) == np.sign(regression["CI_hi"])

best = regression.loc[regression["β, bps/σ"].abs().idxmax()]
print(f"ОТВЕТ B: сильнейший — {best['предиктор']} на {best['h, с']} с: β = {best['β, bps/σ']:+.3f} bps "
      f"на +1σ, CI [{best['CI_lo']:+.3f}, {best['CI_hi']:+.3f}], R² = {best['R²']:.4f} "
      f"→ {best['β/полуспред']:.0%} полуспреда")
print(f"         от микроцены (механика вычтена): β = {best['β от микроцены']:+.3f} bps/σ")
print(f"         CI не содержит 0: {int(regression['CI не содержит 0'].sum())} клеток из {len(regression)}")
display(regression.pivot(index="предиктор", columns="h, с", values="β, bps/σ").round(3))
regression.round(4)

ОТВЕТ B: сильнейший — I_10bps на 30.0 с: β = +0.703 bps на +1σ, CI [+0.592, +0.813], R² = 0.0065 → 10% полуспреда
         от микроцены (механика вычтена): β = -1.984 bps/σ
         CI не содержит 0: 14 клеток из 15


"h, с",0.2,0.5,1.0,5.0,30.0
предиктор,,,,,
I_10bps,0.032,0.073,0.122,0.360,0.703
I_50bps,0.008,0.029,0.052,0.092,0.153
I_touch,0.026,0.054,0.089,0.272,0.527


,предиктор,"h, с","β, bps/σ",CI_lo,CI_hi,β/полуспред,R²,β от микроцены,CI не содержит 0
0,I_touch,0.2,0.0257,0.0232,0.0279,0.0035,0.0012,-4.0331,True
1,I_touch,0.5,0.0543,0.0497,0.0585,0.0074,0.0022,-4.0045,True
2,I_touch,1.0,0.0887,0.0829,0.0946,0.0121,0.0030,-3.9701,True
3,I_touch,5.0,0.2715,0.2478,0.2945,0.0372,0.0056,-3.7872,True
4,I_touch,30.0,0.5265,0.4068,0.6445,0.0721,0.0034,-3.5322,True
5,I_10bps,0.2,0.0324,0.0300,0.0345,0.0044,0.0029,-2.6540,True
6,I_10bps,0.5,0.0732,0.0692,0.0775,0.0100,0.0052,-2.6132,True
7,I_10bps,1.0,0.1223,0.1158,0.1291,0.0167,0.0071,-2.5642,True
8,I_10bps,5.0,0.3596,0.3351,0.3832,0.0492,0.0111,-2.3268,True
9,I_10bps,30.0,0.7026,0.5920,0.8126,0.0962,0.0065,-1.9838,True


##### C. Знаковая точность

Доля случаев, когда знак `Δmid` совпал со знаком имбаланса. Baseline = **50%**. Сэмплы с `Δmid = 0`
исключены — знак не определён (на 0.2 с таких много, см. таблицу выше). Отдельно считаем на сильном
сигнале `|I| > 0.5`: котировки двигают не всегда, а когда сигнал есть.

In [36]:
STRONG_THRESHOLD = 0.5


def sign_accuracy(x, target, threshold):
    """Доля совпавших знаков и N, на котором она посчитана (Δmid = 0 отброшены)."""
    use = np.isfinite(x) & (np.abs(x) >= threshold) & (target != 0)
    if not use.any():
        return np.nan, 0
    return float(np.mean(np.sign(x[use]) == np.sign(target[use]))), int(use.sum())


accuracy = pd.DataFrame([
    {"предиктор": f"I_{name}", "h, с": h,
     "точность": sign_accuracy(predictors[name], drift_from_mid[:, j], 0)[0],
     "N": sign_accuracy(predictors[name], drift_from_mid[:, j], 0)[1],
     f"точность, |I|>{STRONG_THRESHOLD}": sign_accuracy(predictors[name], drift_from_mid[:, j],
                                                        STRONG_THRESHOLD)[0],
     "N сильных": sign_accuracy(predictors[name], drift_from_mid[:, j], STRONG_THRESHOLD)[1]}
    for name in PREDICTORS for j, h in enumerate(HORIZONS_S)
])

best_acc = accuracy.loc[(accuracy["точность"] - 0.5).abs().idxmax()]
print(f"ОТВЕТ C: baseline 50%. Дальше всех от него — {best_acc['предиктор']} на {best_acc['h, с']} с: "
      f"{best_acc['точность']:.1%} (N = {best_acc['N']:,})")
accuracy.round(3)

ОТВЕТ C: baseline 50%. Дальше всех от него — I_10bps на 0.2 с: 52.9% (N = 36,634)


,предиктор,"h, с",точность,N,"точность, |I|>0.5",N сильных
0,I_touch,0.2,0.507,40924,0.520,27621
1,I_touch,0.5,0.497,78511,0.501,53315
2,I_touch,1.0,0.501,109099,0.505,72853
3,I_touch,5.0,0.511,218994,0.523,140909
4,I_touch,30.0,0.519,361628,0.536,228687
5,I_10bps,0.2,0.529,36634,0.543,28063
6,I_10bps,0.5,0.526,71469,0.540,54796
7,I_10bps,1.0,0.523,99558,0.538,75865
8,I_10bps,5.0,0.523,201374,0.536,151232
9,I_10bps,30.0,0.525,337336,0.535,249900


##### Плацебо: предиктор из прошлого

Тот же расчёт, но имбаланс взят на 5 минут **раньше** таргета. Связи быть не должно — β обязана
схлопнуться в ноль. Если не схлопнулась, где-то протёк lookahead, и всё выше недействительно.

In [ ]:
PLACEBO_SHIFT_S = 300
shift = int(PLACEBO_SHIFT_S / SAMPLE_STEP_S)
assert shift < len(sample_ts) // 2, "окно короче плацебо-сдвига — уменьши PLACEBO_SHIFT_S"

rows = []
for name in PREDICTORS:
    past, now = predictors[name][:-shift], drift_from_mid[shift:]
    defined = np.isfinite(past)
    beta_placebo, _ = betas(past[defined], now[defined])

    real_defined = np.isfinite(predictors[name])
    beta_real, _ = betas(predictors[name][real_defined], drift_from_mid[real_defined])
    for j, h in enumerate(HORIZONS_S):
        rows.append({"предиктор": f"I_{name}", "h, с": h,
                     "β реальная": beta_real[j], "β плацебо": beta_placebo[j]})

placebo = pd.DataFrame(rows)
leak = placebo["β плацебо"].abs().max()
real = placebo["β реальная"].abs().max()
print(f"ПЛАЦЕБО (имбаланс сдвинут назад на {PLACEBO_SHIFT_S // 60} мин): "
      f"max|β| = {leak:.3f} bps/σ против {real:.3f} на реальном выравнивании — {leak / real:.0%} от сигнала")
placebo.round(4)

ПЛАЦЕБО (имбаланс сдвинут назад на 5 мин): max|β| = 0.017 bps/σ против 0.703 на реальном выравнивании — 2% от сигнала
